# Merge_all — Build a Unified Dataset for **Maneuvers** (single-boat)

This notebook aggregates **validated maneuver intervals** (from `summary.json`) across all runs into a single, analysis-ready CSV.  
It **does not compare two boats**: each maneuver slice comes from one boat’s log and is annotated with run/rider metadata.

---

## Inputs

- **`summary.json`** — produced by `mainCOG`; for each run it lists validated maneuvers with:
  - `maneuver_index`, `maneuver_type`, `start_time`, `end_time`, `duration`, …
- **Data root** (e.g., `../Data_Sailnjord/Maneuvers`) with folders: /date/person/run/single_file.csv
Each run folder must contain **exactly one CSV**.

- **Helper**: `report_fct.filter_interval(df, start, end)` to clip time windows.

---

## What the code does

1. **Load `summary.json`** and iterate all runs & their maneuver intervals.  
2. **Open the run’s CSV** and **slice** rows to `[start_time, end_time]`.  
3. **Annotate** each sliced row with:
 - `run`, `rider_name`, `boat_name`, `maneuver_index`, `maneuver_type`,  
   `interval_duration`, `start_time`, `end_time`.
4. **Recompute rigging-line features** (when `Line_L/Line_R/Line_C` exist):
 - Sort per-row magnitudes → `Line_R2`(min), `Line_L2`(mid), `Line_C2`(max)  
 - `side_line2 = Line_L2 + Line_R2`  
 - `total_line2 = side_line2 + Line_C2`
5. **Concatenate all slices**, sort by `SecondsSince1970`, and **export**.

---

## Output

- **`all_data.csv`** — long, tidy table of maneuver rows across all runs containing telemetry
(`SecondsSince1970`, `Lat`, `Lon`, `SOG`, `VMG`, `COG`, `TWA`, `TWD`, `TWS`, `Heel_Lwd`, …),
maneuver metadata, and optional derived line metrics (`Line_*2`, `side_line2`, `total_line2`).


In [1]:
# import os
# import json
# import pandas as pd
# import numpy as np
# from report_fct import filter_interval
# import pandas as pd

# def build_csv_from_summary(summary_path, data_root, output_csv="all_data.csv"):
#     # Load the summary JSON
#     with open(summary_path, "r") as f:
#         summary = json.load(f)

#     all_rows = []

#     # Iterate over each run entry in the summary
#     for run_entry in summary:
#         run_name = run_entry["run"]
#         date = run_entry["date"]  # e.g., "08_06"
#         person = run_entry["person"]  # e.g., "Gian"
#         intervals = run_entry["intervals"]

#         # Ensure intervals is a list
#         if not isinstance(intervals, list):
#             print(f"⚠️ Invalid interval data for {run_name}: expected a list, found {type(intervals)}")
#             continue

#         # Build the run folder path using date, person, and run name
#         run_path = os.path.join(data_root, date, person, run_name)
#         # Check if the run folder exists
#         if not os.path.isdir(run_path):
#             print(f"⚠️ Run folder not found for {run_name} at {run_path}")
#             continue

#         # Search for the CSV file in the run folder
#         csv_files = [f for f in os.listdir(run_path) if f.endswith(".csv")]
#         if len(csv_files) != 1:
#             print(f"⚠️ Skipping {run_name}: expected 1 CSV, found {len(csv_files)}")
#             continue

#         # Build the full path to the CSV file
#         csv_path = os.path.join(run_path, csv_files[0])

#         # Read the CSV file into a DataFrame
#         try:
#             df = pd.read_csv(csv_path)
#         except Exception as e:
#             print(f"⚠️ Error reading CSV file {csv_path}: {e}")
#             continue

#         # Process each interval in the run
#         try:
#             print(f"Processing run: {run_name}, total intervals: {len(intervals)}")

#             for i, interval in enumerate(intervals):
#                             m_index = interval["maneuver_index"]
#                             start_m = interval["start_time"]
#                             end_m = interval["end_time"]
                            
#                             # --- GESTION DE LA SUPERPOSITION ---
#                             ref = 6
#                             reference_start = start_m - ref
                            
#                             # Si ce n'est pas la première manœuvre, on vérifie la précédente
#                             if i > 0:
#                                 prev_end = intervals[i-1]["maneuver_time"]+2
#                                 if reference_start < prev_end:
#                                     print(f"⚠️ Overlap detected in {run_name} for maneuver {i}, it's reference time (start - {ref}) is overlapping with the end time of the previous maneuver {i-1}.")
#                                     raise ValueError(f"Overlap detected in {run_name} between maneuver {i} and {i-1}.")
#                             df_interval_complet = filter_interval(df, reference_start, end_m)
#                             df_filtered = df_interval_complet.copy()

#                             # 3. Métadonnées
#                             df_filtered["run"] = run_name
#                             df_filtered["rider_name"] = person
#                             df_filtered["boat_name"] = csv_files[0].replace(".csv", "")
#                             df_filtered["maneuver_type"] = interval["maneuver_type"]
#                             df_filtered["interval_duration"] = interval["duration"]
#                             df_filtered["start_time"] = start_m
#                             df_filtered["end_time"] = end_m

#                             # 4. Marquage 0 vs Index
#                             is_inside_maneuver = (df_filtered['SecondsSince1970'] >= start_m) & \
#                                                 (df_filtered['SecondsSince1970'] <= end_m)
                            
#                             df_filtered["maneuver_index"] = np.where(is_inside_maneuver, m_index, 0)
#                             df_filtered["target_id"] = m_index

#                             all_rows.append(df_filtered)

#         except Exception as e:
#             print(f"❌ Error processing run {run_name}, interval {i + 1}: {e}")
#             continue

#     # Final save
#     if not all_rows:
#         print("❌ No valid data found.")
#         return

#     # Combine all rows into a single DataFrame and save to CSV
#     df_global = pd.concat(all_rows, ignore_index=True)
#     df_global = df_global.sort_values(by='SecondsSince1970', ascending=True)
#     df_global.to_csv(output_csv, index=False)
#     print(f"✅ Global CSV saved to: {output_csv}")

import os, json
import pandas as pd
import numpy as np
from report_fct import filter_interval

def build_csv_from_summary(summary_path, data_root, output_csv="all_data.csv"):
    with open(summary_path, "r") as f:
        summary = json.load(f)

    all_rows = []
    stats = {"valid": 0, "skipped": 0}

    for run in summary:
        run_name, person, date = run["run"], run["person"], run["date"]
        run_path = os.path.join(data_root, date, person, run_name)
        
        if not os.path.isdir(run_path):
            continue

        csv_files = [f for f in os.listdir(run_path) if f.endswith(".csv")]
        if len(csv_files) != 1:
            continue

        df = pd.read_csv(os.path.join(run_path, csv_files[0]))
        intervals = run.get("intervals", [])
        
        for i, interval in enumerate(intervals):
            start_m, end_m = interval["start_time"], interval["end_time"]
            ref_start = start_m - 10
            
            # --- VERIFICATION DES CONDITIONS (OVERLAP) ---
            # 1. Disponibilité dans le fichier CSV
            if ref_start < df['SecondsSince1970'].min():
                print(f"Skipped {run_name} M-{interval['maneuver_index']}: Start too close to file beginning.")
                stats["skipped"] += 1
                continue
                
            # 2. Chevauchement avec la manœuvre précédente
            if i > 0 and ref_start < intervals[i-1]["end_time"]:
                print(f"Skipped {run_name} M-{interval['maneuver_index']}: Overlap with previous maneuver.")
                stats["skipped"] += 1
                continue

            # --- EXTRACTION ET META-DONNEES ---
            df_seg = filter_interval(df, ref_start, end_m).copy()
            
            meta = {
                "run": run_name, "rider_name": person,
                "boat_name": csv_files[0].replace(".csv", ""),
                "maneuver_type": interval["maneuver_type"],
                "interval_duration": interval["duration"],
                "start_time": start_m, "end_time": end_m,
                "target_id": interval["maneuver_index"]
            }
            
            for key, value in meta.items():
                df_seg[key] = value

            # Marquage 0 vs Index (uniquement pendant la manœuvre réelle)
            mask = (df_seg['SecondsSince1970'] >= start_m) & (df_seg['SecondsSince1970'] <= end_m)
            df_seg["maneuver_index"] = np.where(mask, interval["maneuver_index"], 0)

            all_rows.append(df_seg)
            stats["valid"] += 1

    # Finalisation
    if all_rows:
        df_global = pd.concat(all_rows, ignore_index=True).sort_values('SecondsSince1970')
        df_global.to_csv(output_csv, index=False)
        print(f"\n✅ Terminé. Valides: {stats['valid']} | Rejetées: {stats['skipped']}")
    else:
        print("❌ Aucune donnée valide extraite.")

In [2]:
build_csv_from_summary(
    summary_path="summary.json",
    # data_root="../Data_Sailnjord/Maneuvers",
    data_root = "../Data_Sailnjord/Port Camargue June 2025/Maneuvers",
    output_csv="all_data.csv"
)

Skipped 08_06_2025_Run1 M-2: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-3: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-4: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-9: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-10: Overlap with previous maneuver.
Skipped 08_06_2025_Run1 M-12: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-3: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-4: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-5: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-6: Overlap with previous maneuver.
Skipped 08_06_2025_Run2 M-11: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-3: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-4: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-5: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-10: Overlap with previous maneuver.
Skipped 08_06_2025_Run3 M-11: Overlap with previous maneuver.
Skipped 08_06_2025_

Skipped 11_06_2025_Run4 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run4 M-4: Overlap with previous maneuver.
Skipped 11_06_2025_Run4 M-10: Overlap with previous maneuver.
Skipped 11_06_2025_Run4 M-11: Overlap with previous maneuver.
Skipped 11_06_2025_Run5 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run5 M-4: Overlap with previous maneuver.
Skipped 11_06_2025_Run5 M-10: Overlap with previous maneuver.
Skipped 11_06_2025_Run5 M-11: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-2: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-5: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-9: Overlap with previous maneuver.
Skipped 11_06_2025_Run1 M-10: Overlap with previous maneuver.
Skipped 11_06_2025_Run2 M-3: Overlap with previous maneuver.
Skipped 11_06_2025_Run2 M-4: Overlap with previous maneuver.
Skipped 11_06_2025_Run2 M-10: Overlap with previous maneuver.
Skipped 11_06_2025


✅ Terminé. Valides: 135 | Rejetées: 115
